# Embeddings - Concept et similarité

## Pourquoi ce notebook existe

Avant de charger SigLIP (Cycle 2.2) et Arctic Embed (Cycle 2.3) qui produisent des vecteurs de dimension 1024, il faut comprendre **ce qu'est un embedding** et **comment on en mesure la similarité**. Ce notebook est un compagnon pédagogique du module `src/encoders/similarity_utils.py`.

**Discipline** : aucune donnée externe ici. On démontre les concepts sur des **vecteurs synthétiques 2D et N-D**, ce qui suffit pour acquérir l'intuition. Les modèles réels (SigLIP, Arctic) appliquent les mêmes principes en dim 1024.

## Plan

1. **Section 1** - Qu'est-ce qu'un embedding (intuition 2D)
2. **Section 2** - Trois métriques de similarité (cosine, IP, L2) et quand utiliser laquelle
3. **Section 3** - Pourquoi normaliser L2 (équivalence cosine ↔ IP, compatibilité FAISS)
4. **Section 4** - Visualisation t-SNE de clusters synthétiques
5. **Section 5** - Top-k retrieval simple (sans FAISS)

## Place dans le projet

```
Cycle 2 - Représentations vectorielles (P03)
├── Sous-todo 2.1 - Embeddings concept + similarity utils  ← TU ES ICI
├── Sous-todo 2.2 - Vision encoder SigLIP (E1)            → produit data/embeddings/vision_siglip_v1.npy
├── Sous-todo 2.3 - Text encoder Arctic Embed L v2 (E2)   → produit data/embeddings/text_arctic_v1.npy
└── Sous-todo 2.4 - Build indices + sanity check          → utilise les fonctions de ce notebook sur les vrais embeddings
```

**Règle d'or** : R11 - toute normalisation L2 doit avoir un **guard** sur la norme nulle, sous peine de NaN propagé silencieusement. Le module `similarity_utils.py` implémente ce guard une seule fois pour tout le projet.

---

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.encoders.similarity_utils import (
    l2_normalize,
    cosine_similarity,
    inner_product,
    l2_distance,
    top_k_similar,
)

sns.set_theme(style='whitegrid', context='notebook')
RNG = np.random.default_rng(42)
print('Setup OK')

---

## Section 1 - Qu'est-ce qu'un embedding ?

Un **embedding** est une représentation **vectorielle dense** d'un objet (image, texte, produit). L'idée : projeter des objets sémantiquement similaires sur des points proches dans un espace de dimension N.

**Démo 2D** : 4 catégories de produits hypothétiques, chacune représentée par un cluster de points. Plus deux points sont proches, plus les produits qu'ils représentent sont similaires.

In [ ]:
# 4 clusters synthétiques en 2D (centres des cat. + bruit gaussien)
centers = np.array([
    [+2.0, +2.0],   # Books
    [-2.0, +2.0],   # Electronics
    [-2.0, -2.0],   # Toys
    [+2.0, -2.0],   # Clothing
])
labels = ['Books', 'Electronics', 'Toys', 'Clothing']
colors = ['#2563eb', '#16a34a', '#f59e0b', '#dc2626']

n_per_cluster = 30
embeddings_2d = np.vstack([
    c + RNG.normal(scale=0.4, size=(n_per_cluster, 2)) for c in centers
])
labels_arr = np.repeat(labels, n_per_cluster)

fig, ax = plt.subplots(figsize=(8, 6))
for label, color in zip(labels, colors):
    mask = labels_arr == label
    ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], c=color, label=label, s=60, alpha=0.7)
ax.set_title('Embeddings 2D synthétiques - 4 catégories de produits')
ax.set_xlabel('Dimension 0')
ax.set_ylabel('Dimension 1')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---

## Section 2 - Trois métriques de similarité

Pour mesurer à quel point deux embeddings sont similaires :

| Métrique | Formule | Range | Quand utiliser ? |
|---|---|---|---|
| **Cosine** | `cos(a,b) = (a·b) / (\|\|a\|\| · \|\|b\|\|)` | [-1, 1] | Quand la **direction** porte le sens, pas la magnitude (cas standard SigLIP/Arctic) |
| **Inner product** | `ip(a,b) = a·b` | (-∞, +∞) | Équivalent à cosine **après** normalisation L2. Plus rapide en FAISS (METRIC_INNER_PRODUCT) |
| **L2 distance** | `\|\|a-b\|\|` | [0, +∞) | Quand on raisonne en **distance géométrique**. Sur vecteurs L2-normalisés, équivalent monotone à cosine |

**Démo** : prenons un vecteur "requête" et calculons sa similarité avec un vecteur de chaque catégorie.

In [ ]:
import pandas as pd

# Query proche du cluster Books
query = np.array([2.1, 1.9])

# Un représentant de chaque cluster
samples = centers  # shape (4, 2)

results = pd.DataFrame({
    'category': labels,
    'point': [tuple(p) for p in samples],
    'cosine_sim': cosine_similarity(query, samples),
    'inner_product': inner_product(query, samples),
    'l2_distance': l2_distance(query, samples),
})
results

**Lecture** :
- Cosine identifie correctement Books comme le plus proche (0,99)
- Inner product donne le même classement que cosine ICI car les vecteurs ont des magnitudes similaires (≈ 2,83)
- L2 distance classe l'inverse : plus c'est petit, plus c'est proche → Books a la plus petite L2

**Trois métriques, trois sens, mais classement cohérent quand les vecteurs sont normalisés**. C'est pour ça que SigLIP, Arctic, FAISS travaillent par défaut sur des vecteurs **pré-normalisés L2**.

---

## Section 3 - Pourquoi normaliser L2 ?

Comparons cosine et inner product sur des vecteurs **non normalisés** vs **normalisés**.

**Cas pathologique** : un vecteur très grand `[10, 10]` ET un vecteur très petit `[0.1, 0.1]`. Tous deux pointent vers Books (direction (1,1)/√2), mais leur magnitude diffère de 100×.

In [ ]:
big_books = np.array([10.0, 10.0])
small_books = np.array([0.1, 0.1])
query2 = np.array([2.0, 2.0])

print('=== Sans normalisation ===')
print(f'cosine_sim(query, big_books)   = {cosine_similarity(query2, big_books):.4f}')
print(f'cosine_sim(query, small_books) = {cosine_similarity(query2, small_books):.4f}')
print(f'inner_product(query, big_books)   = {inner_product(query2, big_books):.4f}')
print(f'inner_product(query, small_books) = {inner_product(query2, small_books):.4f}')

print('\n=== Après normalisation L2 ===')
q_n = l2_normalize(query2)
big_n = l2_normalize(big_books)
small_n = l2_normalize(small_books)
print(f'inner_product(L2(query), L2(big_books))   = {inner_product(q_n, big_n):.4f}')
print(f'inner_product(L2(query), L2(small_books)) = {inner_product(q_n, small_n):.4f}')

**Lecture** :
- **Cosine** : ignore la magnitude → big et small ont la même similarité 1.0 avec query (même direction)
- **Inner product brut** : 40 vs 0.4 → big écrase totalement small alors que sémantiquement ils sont équivalents
- **Inner product après L2 normalize** : même valeur (1.0) qu'avec cosine - c'est l'équivalence mathématique

**Conséquence pratique pour FAISS** : on pré-normalise tous les vecteurs UNE FOIS au build de l'index, puis on utilise `METRIC_INNER_PRODUCT` (plus rapide que `METRIC_COSINE` qui n'existe pas en FAISS - il faudrait re-normaliser à chaque query).

---

## Section 4 - Visualisation t-SNE de clusters

Sur des embeddings réels en dim 1024, on ne peut pas visualiser directement. **t-SNE** projette en 2D en préservant la structure locale (clusters proches restent proches).

Démo : 4 catégories synthétiques en dim 1024, viz t-SNE en 2D.

In [ ]:
from sklearn.manifold import TSNE

# 4 clusters en dim 1024 (centres aléatoires + bruit)
DIM = 1024
N_PER_CAT = 50
centers_hd = RNG.normal(size=(4, DIM))
embeddings_hd = np.vstack([
    c + RNG.normal(scale=0.5, size=(N_PER_CAT, DIM)) for c in centers_hd
])
labels_hd = np.repeat(labels, N_PER_CAT)

# Normalisation L2 (pratique standard avant t-SNE sur des embeddings sémantiques)
embeddings_hd = l2_normalize(embeddings_hd)

# Projection t-SNE 2D
print(f'Projection t-SNE de {embeddings_hd.shape[0]} vecteurs dim {DIM} → 2D…')
tsne = TSNE(n_components=2, perplexity=15, random_state=42, init='pca')
proj = tsne.fit_transform(embeddings_hd)

fig, ax = plt.subplots(figsize=(8, 6))
for label, color in zip(labels, colors):
    mask = labels_hd == label
    ax.scatter(proj[mask, 0], proj[mask, 1], c=color, label=label, s=60, alpha=0.7)
ax.set_title(f't-SNE de {DIM}-dim embeddings (4 clusters synthétiques)')
ax.set_xlabel('t-SNE dim 0')
ax.set_ylabel('t-SNE dim 1')
ax.legend()
plt.tight_layout()
plt.show()

**Lecture** : t-SNE retrouve les 4 clusters bien séparés. C'est l'**outil de validation visuel** qu'on utilisera au Cycle 2.4 (sanity check des vrais embeddings SigLIP) et au Cycle 8 (Model Cards explainability).

---

## Section 5 - Top-k retrieval (sans FAISS)

Pour des corpus < 100k vecteurs, `top_k_similar` du module `similarity_utils` suffit (pas besoin de FAISS). Démo : retrouver les 3 voisins les plus proches d'un query parmi les 200 vecteurs synthétiques.

In [ ]:
# Query : un point au centre du cluster Books
query_hd = centers_hd[0] + RNG.normal(scale=0.1, size=DIM)
query_hd = l2_normalize(query_hd)

idx, scores = top_k_similar(query_hd, embeddings_hd, k=5, metric='cosine')
print('Top-5 voisins (cosine descendant) :')
for rank, (i, s) in enumerate(zip(idx, scores)):
    print(f'  rank {rank+1} : index={i:3d}, label={labels_hd[i]:12s}, cosine={s:.4f}')

# Vérification : tous les top-5 devraient être Books (cluster d'où vient le query)
n_books = sum(labels_hd[i] == 'Books' for i in idx)
print(f'\nValidation : {n_books}/5 top voisins sont Books - attendu : 5/5 ✅' if n_books == 5 else f'⚠️ {n_books}/5 - bruit trop élevé ?')

---

## Section 6 - Pour aller plus loin

- **Cycle 2.2** : appliquer SigLIP sur ~26 M images d'items meta → vrais embeddings vision
- **Cycle 2.3** : appliquer Arctic Embed sur title+description → vrais embeddings texte
- **Cycle 2.4** : sanity check (notebook similaire à celui-ci, mais sur les vrais embeddings)
- **Cycle 4** : passage à FAISS quand le corpus dépasse 100k vecteurs (HNSW, IVF_PQ)

**Code source** : `src/encoders/similarity_utils.py` (5 fonctions documentées + 22 tests dans `tests/test_similarity_utils.py`)

**Décisions / règles** : R11 (guard normalisation L2), R3 (anti-leakage : ces fonctions sont stateless donc safe), D-009 (architecture cible RAG-grounded)